# 0.0 - Imports

## 0.1 - Importa as bibliotecas

In [16]:
import os
import pandas as pd
import plotly.express as px

from dotenv import load_dotenv
from sqlalchemy import create_engine, text

## 0.2 - Carrega as credenciais do .env

In [2]:
load_dotenv()

True

## 0.3 - Conexão utilizando o .env

In [3]:
user = os.getenv('DB_USER')
password = os.getenv('DB_PASS')
host = os.getenv('DB_HOST')
dataset = os.getenv('DB_NAME')

url = create_engine(f"mysql+pymysql://{user}:{password}@{host}/{dataset}")

## 0.4 - Teste de Conexão

In [4]:
with url.connect() as conn:
    result = conn.execute(text("SELECT VERSION();"))
    print(result.fetchone())

('8.0.31-google',)


# 1.0 - Execução das consultas

In [5]:
query_imdb = text('SELECT * FROM IMDB_movies im')

In [6]:
#Carrega no pandas a tabela IMDB Movies
df_raw_imdb = pd.read_sql_query(query_imdb, con=url)

In [10]:
#Verificação dos tipos do DataFrame e tamanho
print(df_raw_imdb.dtypes)
print(df_raw_imdb.shape)

Id                   int64
Title                  str
Genre                  str
Director               str
Actors                 str
Year                 int64
Runtime              int64
Rating             float64
Votes                int64
RevenueMillions    float64
Metascore          float64
dtype: object
(1000, 11)


In [9]:
#Verifica se datasets estão de acordo com exemplos
df_raw_imdb.head(5)

,Id,Title,Genre,Director,Actors,Year,Runtime,Rating,Votes,RevenueMillions,Metascore
0,1,Guardians of the Galaxy,"Action,Adventure,Sci-Fi",James Gunn,"Chris Pratt, Vin Diesel, Bradley Cooper, Zoe S...",2014,121,8.0,757074,333.0,76.0
1,2,Prometheus,"Adventure,Mystery,Sci-Fi",Ridley Scott,"Noomi Rapace, Logan Marshall-Green, Michael Fa...",2012,124,7.0,485820,126.0,65.0
2,3,Split,"Horror,Thriller",M. Night Shyamalan,"James McAvoy, Anya Taylor-Joy, Haley Lu Richar...",2016,117,7.0,157606,138.0,62.0
3,4,Sing,"Animation,Comedy,Family",Christophe Lourdelet,"Matthew McConaughey,Reese Witherspoon, Seth Ma...",2016,108,7.0,60545,270.0,59.0
4,5,Suicide Squad,"Action,Adventure,Fantasy",David Ayer,"Will Smith, Jared Leto, Margot Robbie, Viola D...",2016,123,6.0,393727,325.0,40.0


# 2.0 - Análise do Dataset a partir da bilheteria como principal problema de negócio

### 2.1 - Hipotese 1 - Filmes com mais votos geram maior bilheteria?

**VERDADEIRO** - Existe uma relação entre o aumento de votos e a bilheteria

In [ ]:
#Grafico de Scatter Plot para Observar a relação entre Votes e RevenueMillions
fig = px.scatter(
    df_raw_imdb, 
    x='RevenueMillions', 
    y ='Votes', 
    trendline="ols", 
    trendline_color_override="red", 
    title="Relação entre Bilheteria e Número de Votos")
fig.show()

### 2.2 - Hipotese 2 - Gêneros de ação e aventura (mais populares) geram maior bilheteria, independente da avaliação?

**VERDADEIRO** - Generos de Ação e Aventura estão entre as maiores bilheterias independente da avaliação da critica

In [74]:
# Mantem apenas o primeiro titulo do Genero
df_genero_filtrado = df_raw_imdb.copy()
df_genero_filtrado['Genre'] = df_raw_imdb['Genre'].str.split(',').str[0]

#Filtros para total de filmes e bilheteria total de generos
df_genero_filtrado = df_genero_filtrado.dropna(subset=['RevenueMillions'])
contagem = df_genero_filtrado['Genre'].value_counts()
generos_validos = contagem[contagem >= 10].index

df_genero_filtrado = df_genero_filtrado[df_genero_filtrado['Genre'].isin(generos_validos)]

ordenação = (
    df_genero_filtrado.groupby('Genre')['RevenueMillions']
    .median()
    .sort_values()
    .index.tolist()
)

# Cria o grafico de barra comparando a bilheteria pelo Genero Principal
fig = px.box(
    df_genero_filtrado, 
    x='RevenueMillions', 
    y='Genre',
    category_orders={'Genre': ordenação}
)
fig.show()

### 2.3 - Hipotese 3 - Filmes com Metascore alto tendem a ter menor bilheteria do que filmes com Metascore baixo?

**VERDADEIRO** - Filmes com Metascore maiores tendem a ter menor bilheteria

In [ ]:
# Cria uma cópia do Dataset Original
df_metascore_filtrado = df_raw_imdb.copy()

#Cria faixas de Metascore para analise agrupada diminuindo a granularidade
df_metascore_filtrado['Metascore_Faixa'] = pd.cut(df_metascore_filtrado['Metascore'],
                                                  bins=[0, 20, 40, 60, 80, 100],
                                                  labels=['0-20', '20-40', '40-60', '60-80', '80-100'])

# Agrupa por Metascore
df_metascore_filtrado = df_metascore_filtrado.groupby(['Metascore_Faixa']).agg(
    RevenueMillions=('RevenueMillions', 'sum'),
    Rating=('Rating', 'mean')
).reset_index().sort_values(by='RevenueMillions', ascending=True)

df_metascore_filtrado['Rating'] = df_metascore_filtrado['Rating'].round(1)

# Cria o grafico de barra comparando a bilheteria pelo Genero Principal
fig = px.bar(
    df_metascore_filtrado, 
    x='Metascore_Faixa', 
    y='RevenueMillions',
    color='Rating',
    color_continuous_scale='Blues',
    category_orders={'Metascore_Faixa': ['0-20', '20-40', '40-60', '60-80', '80-100']}
)
fig.show()